<img src="./banner.png" width="100%">


Architecture: Multi-Agent Turn-Based Conversation Loop

This implementation dictates the logic for a multi-agent Large Language Model (LLM) interaction. The architecture relies on dynamic state management and iterative payload construction to maintain continuous context across distinct character personas.

Execution Methodology:

    Persona Initialization: Define isolated system prompts to establish the specific behavioral rules and identity for each agent.

    Global State Management: Initialize a universal transcript list to serve as the persistent memory, storing the chronological conversation history.

    Agent Routing & Mapping: Establish an ordered list of agents (to dictate the turn sequence) and construct a prompt_map dictionary linking each agent string to its specific system prompt data.

    Dynamic Context Generation (generate_turn): Build a generalized function that dynamically constructs the API payload for every call by merging the active agent's system prompt with the global transcript.

    Single-Round Execution: Implement a for loop to iterate through the agent roster, executing one complete conversational turn per character and appending the output to the master transcript.

    Multi-Round Session Control: Encapsulate the execution logic within a bounded while loop to sustain continuous multi-round interactions, organically building the shared conversational context over time.

In [ ]:
import os
import requests
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
requests.get("http://localhost:11434/").content


In [ ]:
base_url = "http://localhost:11434/v1"
api_key = "llama-api"

model = "llama3.2:latest"

deadpool_prompt = [{"role":"system", "content":"You are Deadpool from the Marvel comics, youre responses are as close to the Deadpool personality as possible"}]

captain_america_prompt = [{"role":"system", "content":"You are Captain America from the Marvel comics, youre responses are as close to the Captain America personality as possible"}]

ironman_prompt = [{"role":"system", "content":"You are Ironman from the Marvel comics, youre responses are as close to the Ironman personality as possible"}]

messages = [{"role":"user", "content":"what is 1 + 1?"}]

In [ ]:
!ollama pull llama3.2:latest

In [ ]:
ollama = OpenAI(base_url=base_url, api_key=api_key)

In [ ]:
transcript =["system_event: The Avengers and Deadpool have just walked into a room. Introduce yourself."]

In [ ]:
agents = ["deadpool", "captain_america", "ironman"]

In [ ]:
response = ollama.chat.completions.create(model=model,messages=messages)
display(Markdown(response.choices[0].message.content))


In [ ]:
prompt_map ={
    "deadpool": deadpool_prompt,
    "captain_america": captain_america_prompt,
    "ironman": ironman_prompt
}

In [ ]:
def generate_turn(agent_name, current_prompt):
    
    messages_payload = list(current_prompt)
    
    # Concatenate the entire list of strings into a single text block separated by line breaks
    formatted_history = "\n".join(transcript)
    
    # Create a  framing instruction forcing the model to view the history objectively
    framing_instruction = (
        f"Here is the transcript of the conversation so far:\n\n"
        f"{formatted_history}\n\n"
        f"Based on the transcript above, generate the next response. You are acting strictly as {agent_name}."
        f"Conversations are kept short , no lengthy reads allowed."
    )
    
    # Append the framed history as a single user prompt
    messages_payload.append({"role": "user", "content": framing_instruction})
        
    # 3. Send this complete package to the model
    response = ollama.chat.completions.create(
        model=model, 
        messages=messages_payload
    )
    
    # 4. Return the raw text of the response
    return response.choices[0].message.content

In [ ]:
for agent in agents:
    # 1. Look up the correct system prompt list using the dictionary
    current_prompt = prompt_map[agent]
    
    # 2. Execute the function and catch the returned text
    raw_reply = generate_turn(agent, current_prompt)
    
    # 3. Prepend the character's name so the model knows who is speaking
    attributed_reply = agent + ": " + raw_reply
    
    # 4. Save this formatted string to the master history list
    transcript.append(attributed_reply)
    
    # 5. Output the result to the console
    print(attributed_reply)
    print("--------------------------------------------------")

In [ ]:
max_rounds = 3
current_round = 0

In [ ]:
while current_round < max_rounds:

    print(f"\n--- STARTING ROUND {current_round + 1} ---")  

    for agent in agents:
        # 1. Look up the correct system prompt list using the dictionary
        current_prompt = prompt_map[agent]
        
        # 2. Execute the function and catch the returned text
        raw_reply = generate_turn(agent, current_prompt)
        
        # 3. Prepend the character's name so the model knows who is speaking
        attributed_reply = agent + ": " + raw_reply
        
        # 4. Save this formatted string to the master history list
        transcript.append(attributed_reply)
        
        # 5. Output the result to the console
        display(Markdown(attributed_reply))
        print("--------------------------------------------------")

    current_round += 1

print("\n--- CONVERSATION FINISHED ---")